In [1]:
# ============================================================================
# DIAGNOSTIC QUERIES - FIND MISSING OBSERVATION DATES
# ============================================================================

from pyspark.sql import functions as F

print("=" * 80)
print("DIAGNOSTIC: FINDING MISSING OBSERVATION DATES")
print("=" * 80)

# ============================================================================
# CHECK 1: What's in the source tables?
# ============================================================================

print("\n1. CHECKING SOURCE TABLES")
print("-" * 80)

# Check predictions table
print("\nA. gold_credit_score_predictions:")
predictions_check = spark.table("gold_credit_score_predictions")
predictions_dates = predictions_check.groupBy("observation_date").agg(
    F.count("*").alias("records"),
    F.countDistinct("retailer_id").alias("unique_retailers")
).orderBy("observation_date")

predictions_dates.show(20, False)

print(f"\nTotal records: {predictions_check.count():,}")
print(f"Unique dates: {predictions_check.select('observation_date').distinct().count()}")

# Check credit limits table
print("\nB. gold_credit_limits:")
limits_check = spark.table("gold_credit_limits")
limits_dates = limits_check.groupBy("observation_date").agg(
    F.count("*").alias("records"),
    F.countDistinct("retailer_id").alias("unique_retailers")
).orderBy("observation_date")

limits_dates.show(20, False)

# Check features table
print("\nC. gold_credit_scoring_features:")
features_check = spark.table("gold_credit_scoring_features")
features_dates = features_check.groupBy("observation_date").agg(
    F.count("*").alias("records"),
    F.countDistinct("retailer_id").alias("unique_retailers")
).orderBy("observation_date")

features_dates.show(20, False)

# ============================================================================
# CHECK 2: What's in the fact table?
# ============================================================================

print("\n" + "=" * 80)
print("2. CHECKING FACT TABLE")
print("-" * 80)

fact_check = spark.table("gold_fact_credit_scoring")

print(f"\nTotal records in fact table: {fact_check.count():,}")
print(f"Unique observation dates: {fact_check.select('date_key').distinct().count()}")

fact_dates = fact_check.groupBy("date_key").agg(
    F.count("*").alias("records"),
    F.countDistinct("retailer_key").alias("unique_retailers"),
    F.sum("final_credit_limit").alias("total_exposure")
).orderBy("date_key")

print("\nBreakdown by date:")
fact_dates.show(20, False)

# ============================================================================
# CHECK 3: Are records being dropped in the joins?
# ============================================================================

print("\n" + "=" * 80)
print("3. CHECKING FOR JOIN ISSUES")
print("-" * 80)

# Count before joins
pred_count = predictions_check.count()
limits_count = limits_check.count()
features_count = features_check.count()

print(f"\nRecords before join:")
print(f"  Predictions: {pred_count:,}")
print(f"  Credit Limits: {limits_count:,}")
print(f"  Features: {features_count:,}")

# Check for mismatches
print("\nChecking for missing matches...")

# Predictions not in limits
missing_in_limits = predictions_check.join(
    limits_check,
    ["retailer_id", "observation_date"],
    "left_anti"
).count()

print(f"  Predictions missing in credit_limits: {missing_in_limits:,}")

# Predictions not in features
missing_in_features = predictions_check.join(
    features_check,
    ["retailer_id", "observation_date"],
    "left_anti"
).count()

print(f"  Predictions missing in features: {missing_in_features:,}")

# ============================================================================
# CHECK 4: Original training data
# ============================================================================

print("\n" + "=" * 80)
print("4. CHECKING ORIGINAL TRAINING DATA")
print("-" * 80)

# This is what you originally created
original_features = spark.table("gold_credit_scoring_features")

print(f"\nTotal records in original features: {original_features.count():,}")

original_dates = original_features.groupBy("observation_date").agg(
    F.count("*").alias("records")
).orderBy("observation_date")

print("\nAll observation dates in original data:")
original_dates.show(20, False)

# ============================================================================
# CHECK 5: Test vs Train Split Issue
# ============================================================================

print("\n" + "=" * 80)
print("5. CHECKING IF TEST/TRAIN SPLIT AFFECTED DATA")
print("-" * 80)

train_cutoff = "2024-09-15"

before_cutoff = original_features.filter(
    F.col("observation_date") < train_cutoff
).count()

after_cutoff = original_features.filter(
    F.col("observation_date") >= train_cutoff
).count()

print(f"\nBefore {train_cutoff}: {before_cutoff:,} records")
print(f"After {train_cutoff}: {after_cutoff:,} records")

# ============================================================================
# RECOMMENDATION
# ============================================================================

print("\n" + "=" * 80)
print("DIAGNOSIS & RECOMMENDATIONS")
print("=" * 80)

if fact_check.select('date_key').distinct().count() < 6:
    print("\n⚠️  ISSUE DETECTED: Fact table has fewer dates than expected")
    print("\nPossible causes:")
    print("  1. Star schema was built from test data only (after 2024-09-15)")
    print("  2. Joins are dropping records due to missing keys")
    print("  3. Credit limits or predictions tables are incomplete")
    
    print("\n✅ SOLUTION:")
    print("  Rebuild fact table using ALL observation dates from predictions table")
    print("  See the corrected code below...")
else:
    print("\n✓ Fact table looks good!")
    print(f"  Contains {fact_check.select('date_key').distinct().count()} observation dates")

# ============================================================================
# SHOW WHAT SHOULD BE IN FACT TABLE
# ============================================================================

print("\n" + "=" * 80)
print("EXPECTED vs ACTUAL")
print("=" * 80)

expected_dates = predictions_check.select("observation_date").distinct().count()
actual_dates = fact_check.select("date_key").distinct().count()

print(f"\nExpected observation dates: {expected_dates}")
print(f"Actual dates in fact table: {actual_dates}")
print(f"Missing dates: {expected_dates - actual_dates}")

if expected_dates > actual_dates:
    print("\n❌ DATA LOSS DETECTED!")
    print("\nMissing dates:")
    
    all_pred_dates = predictions_check.select(
        F.col("observation_date").alias("date")
    ).distinct()
    
    all_fact_dates = fact_check.select(
        F.col("date_key").alias("date")
    ).distinct()
    
    missing = all_pred_dates.join(all_fact_dates, "date", "left_anti")
    missing.show(20, False)
    
    print("\n🔧 ACTION REQUIRED: Rebuild fact table to include these dates")

print("\n" + "=" * 80)

StatementMeta(, 988cc853-e3a2-45b7-be1f-ff17215413e9, 3, Finished, Available, Finished)

DIAGNOSTIC: FINDING MISSING OBSERVATION DATES

1. CHECKING SOURCE TABLES
--------------------------------------------------------------------------------

A. gold_credit_score_predictions:
+----------------+-------+----------------+
|observation_date|records|unique_retailers|
+----------------+-------+----------------+
|2024-09-15      |1122   |1122            |
|2024-09-30      |1540   |1540            |
|2024-10-15      |2006   |2006            |
+----------------+-------+----------------+


Total records: 4,668
Unique dates: 3

B. gold_credit_limits:
+----------------+-------+----------------+
|observation_date|records|unique_retailers|
+----------------+-------+----------------+
|2024-09-15      |1122   |1122            |
|2024-09-30      |1540   |1540            |
|2024-10-15      |2006   |2006            |
+----------------+-------+----------------+


C. gold_credit_scoring_features:
+----------------+-------+----------------+
|observation_date|records|unique_retailers|
+--------

In [1]:
# ============================================================================
# STAR SCHEMA SEMANTIC MODEL FOR CREDIT SCORING DASHBOARD
# Create fact and dimension tables optimized for Power BI reporting
# ============================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

print("=" * 80)
print("BUILDING STAR SCHEMA SEMANTIC MODEL FOR CREDIT SCORING")
print("=" * 80)

# ============================================================================
# LOAD SOURCE DATA
# ============================================================================

print("\nLoading source data...")
predictions = spark.table("gold_credit_score_predictions")
credit_limits = spark.table("gold_credit_limits")
features = spark.table("gold_credit_scoring_features")
retailers_master = spark.table("Silver.dbo.retailers")
transactions = spark.table("Silver.dbo.silver_retailer_transactions")

# ============================================================================
# DIMENSION 1: DIM_DATE (Date Dimension)
# ============================================================================

print("\n" + "=" * 80)
print("CREATING DIM_DATE")
print("=" * 80)

# Get all unique dates from transactions and observation dates
all_dates = transactions.select(F.col("order_date").alias("date")) \
    .union(predictions.select(F.col("observation_date").alias("date"))) \
    .distinct()

dim_date = all_dates.select(
    F.col("date").alias("date_key"),
    F.year("date").alias("year"),
    F.quarter("date").alias("quarter"),
    F.month("date").alias("month"),
    F.dayofmonth("date").alias("day"),
    F.dayofweek("date").alias("day_of_week"),
    F.weekofyear("date").alias("week_of_year"),
    F.date_format("date", "MMMM").alias("month_name"),
    F.date_format("date", "EEEE").alias("day_name"),
    F.concat(
        F.lit("Q"), F.quarter("date"), F.lit(" "), F.year("date")
    ).alias("quarter_name"),
    F.concat(
        F.date_format("date", "MMM"), F.lit(" "), F.year("date")
    ).alias("month_year"),
    F.when(F.dayofweek("date").isin([1, 7]), "Weekend").otherwise("Weekday").alias("is_weekend"),
)

# Save dimension
dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_date")

print(f"✓ Created DIM_DATE with {dim_date.count():,} records")

# ============================================================================
# DIMENSION 2: DIM_RETAILER (Retailer Dimension - SCD Type 1)
# ============================================================================

print("\n" + "=" * 80)
print("CREATING DIM_RETAILER")
print("=" * 80)

dim_retailer = retailers_master.select(
    F.col("retailer_id").alias("retailer_key"),
    F.col("business_name"),
    F.col("owner_name"),
    F.col("owner_age"),
    F.col("owner_gender"),
    F.col("phone_number"),
    F.col("email"),
    F.col("shop_type"),
    F.col("state"),
    F.col("urbanization_level"),
    F.col("latitude"),
    F.col("longitude"),
    F.col("years_in_business"),
    F.col("months_in_business"),
    F.col("num_employees"),
    F.col("has_business_registration"),
    F.col("mobile_money_pattern"),
    F.col("monthly_mobile_money_txns"),
    F.col("credit_segment"),
    F.col("credit_limit").alias("original_credit_limit"),
    F.col("is_defaulter"),
    F.col("onboarding_date"),
    F.col("account_status"),
    
    # Derived attributes
    F.when(F.col("owner_age") < 30, "18-29")
     .when(F.col("owner_age") < 40, "30-39")
     .when(F.col("owner_age") < 50, "40-49")
     .otherwise("50+").alias("age_group"),
    
    F.when(F.col("years_in_business") < 2, "New (0-2 years)")
     .when(F.col("years_in_business") < 5, "Established (2-5 years)")
     .when(F.col("years_in_business") < 10, "Mature (5-10 years)")
     .otherwise("Veteran (10+ years)").alias("business_maturity"),
    
    F.when(F.col("num_employees") == 1, "Solo")
     .when(F.col("num_employees") <= 3, "Small (2-3)")
     .when(F.col("num_employees") <= 10, "Medium (4-10)")
     .otherwise("Large (10+)").alias("business_size"),
    
    F.when(F.col("monthly_mobile_money_txns") >= 20, "Heavy (20+)")
     .when(F.col("monthly_mobile_money_txns") >= 10, "Frequent (10-19)")
     .when(F.col("monthly_mobile_money_txns") >= 5, "Regular (5-9)")
     .when(F.col("monthly_mobile_money_txns") >= 1, "Light (1-4)")
     .otherwise("None").alias("mobile_money_frequency"),
    
    # Calculate total months in business
    (F.col("years_in_business") * 12 + F.col("months_in_business")).alias("total_months_in_business"),
    
    # Region grouping (you can customize based on Nigerian regions)
    F.when(F.col("state").isin(["Lagos", "Ogun", "Oyo", "Osun", "Ondo", "Ekiti"]), "South West")
     .when(F.col("state").isin(["Rivers", "Delta", "Bayelsa", "Edo", "Cross River", "Akwa Ibom"]), "South South")
     .when(F.col("state").isin(["Enugu", "Anambra", "Imo", "Abia", "Ebonyi"]), "South East")
     .when(F.col("state").isin(["Kano", "Kaduna", "Katsina", "Jigawa", "Sokoto", "Kebbi", "Zamfara"]), "North West")
     .when(F.col("state").isin(["Borno", "Yobe", "Bauchi", "Gombe", "Adamawa", "Taraba"]), "North East")
     .when(F.col("state").isin(["Niger", "Kwara", "Kogi", "Benue", "Plateau", "Nasarawa", "FCT"]), "North Central")
     .otherwise("Other").alias("region"),
).dropDuplicates(["retailer_key"])

# Save dimension
dim_retailer.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_retailer")

print(f"✓ Created DIM_RETAILER with {dim_retailer.count():,} records")

# ============================================================================
# DIMENSION 3: DIM_RISK_TIER (Risk Tier Dimension)
# ============================================================================

print("\n" + "=" * 80)
print("CREATING DIM_RISK_TIER")
print("=" * 80)

dim_risk_tier = spark.createDataFrame([
    (1, "Platinum", 750, 850, "Excellent", 0.001, "APPROVE", "#FFD700", 1),
    (2, "Gold", 650, 749, "Very Good", 0.005, "APPROVE", "#C0C0C0", 2),
    (3, "Silver", 550, 649, "Good", 0.010, "APPROVE_WITH_MONITORING", "#CD7F32", 3),
    (4, "Bronze", 450, 549, "Fair", 0.030, "CONDITIONAL", "#8B4513", 4),
    (5, "Copper", 0, 449, "Poor", 0.300, "DECLINE", "#B87333", 5),
], [
    "tier_key", "tier_name", "min_score", "max_score", "tier_description", 
    "expected_default_rate", "credit_decision", "tier_color", "tier_rank"
])

# Save dimension
dim_risk_tier.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_risk_tier")

print(f"✓ Created DIM_RISK_TIER with {dim_risk_tier.count():,} records")

# ============================================================================
# DIMENSION 4: DIM_CREDIT_PRODUCT (Credit Product/Limit Tiers)
# Note: This is a lookup dimension - no direct FK, used for filtering only
# ============================================================================

print("\n" + "=" * 80)
print("CREATING DIM_CREDIT_PRODUCT (Lookup Dimension)")
print("=" * 80)

dim_credit_product = spark.createDataFrame([
    (1, "Micro Credit", 0, 50000, "Entry-level credit for new customers"),
    (2, "Small Credit", 50001, 150000, "Small business credit"),
    (3, "Medium Credit", 150001, 500000, "Growing business credit"),
    (4, "Large Credit", 500001, 1000000, "Established business credit"),
    (5, "Premium Credit", 1000001, 2000000, "Premium customer credit"),
], ["product_key", "product_name", "min_limit", "max_limit", "product_description"])

# Save dimension
dim_credit_product.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_credit_product")

print(f"✓ Created DIM_CREDIT_PRODUCT with {dim_credit_product.count():,} records")
print("  Note: Lookup dimension - use final_credit_limit ranges for filtering")

# ============================================================================
# SINGLE FACT TABLE: FACT_CREDIT_SCORING (Consolidated)
# ============================================================================

print("\n" + "=" * 80)
print("CREATING FACT_CREDIT_SCORING (Consolidated Fact Table)")
print("=" * 80)

# Join all data sources
fact_credit_scoring = predictions.alias("pred") \
    .join(credit_limits.alias("lim"), 
          ["retailer_id", "observation_date"], "inner") \
    .join(features.select(
            "retailer_id", "observation_date",
            "on_time_rate_recent", "on_time_rate_medium", "on_time_rate_lifetime",
            "avg_days_late_recent", "avg_days_late_medium", "avg_days_late_lifetime",
            "late_rate_recent", "late_rate_medium",
            "max_days_late_recent", "max_days_late_lifetime",
            "txn_count_recent", "txn_count_medium", "txn_count_lifetime",
            "avg_order_value_recent", "avg_order_value_medium", "avg_order_value_lifetime",
            "total_value_recent", "total_value_lifetime",
            "customer_tenure_days", "days_since_last_order",
            "payment_deterioration_ratio", "late_rate_change",
            "txn_velocity_ratio", "on_time_improvement",
            "payment_consistency", "credit_utilization",
            "unique_categories", "avg_orders_per_month"
          ).alias("feat"),
          ["retailer_id", "observation_date"], "inner")

# Create surrogate key
fact_credit_scoring = fact_credit_scoring.withColumn(
    "fact_key",
    F.monotonically_increasing_id()
)

# Map to risk tier key
fact_credit_scoring = fact_credit_scoring.join(
    dim_risk_tier.select("tier_name", "tier_key"),
    F.col("pred.predicted_tier") == F.col("tier_name"),
    "left"
).withColumnRenamed("tier_key", "risk_tier_key")

# Map to credit product key
fact_credit_scoring = fact_credit_scoring.withColumn(
    "credit_product_key",
    F.when(F.col("final_credit_limit") <= 50000, 1)
     .when(F.col("final_credit_limit") <= 150000, 2)
     .when(F.col("final_credit_limit") <= 500000, 3)
     .when(F.col("final_credit_limit") <= 1000000, 4)
     .when(F.col("final_credit_limit") > 1000000, 5)
     .otherwise(1)
)

# Select final consolidated fact table
fact_credit_scoring = fact_credit_scoring.select(
    # Keys
    F.col("fact_key"),
    F.col("pred.retailer_id").alias("retailer_key"),
    F.col("pred.observation_date").alias("date_key"),
    F.col("risk_tier_key"),
    F.col("credit_product_key"),
    
    # ========================================================================
    # CREDIT SCORE MEASURES
    # ========================================================================
    F.col("pred.credit_score").alias("actual_credit_score"),
    F.col("pred.predicted_score").alias("predicted_credit_score"),
    F.abs(F.col("pred.credit_score") - F.col("pred.predicted_score")).alias("prediction_error"),
    
    # ========================================================================
    # CREDIT LIMIT MEASURES
    # ========================================================================
    F.col("lim.base_credit_limit"),
    F.col("lim.final_credit_limit"),
    F.col("lim.total_multiplier"),
    F.col("lim.history_multiplier"),
    F.col("lim.payment_bonus"),
    F.col("lim.maturity_bonus"),
    F.col("lim.shop_type_multiplier"),
    F.col("lim.location_multiplier"),
    
    # ========================================================================
    # RISK MEASURES
    # ========================================================================
    F.col("lim.expected_default_rate"),
    F.col("lim.expected_loss"),
    F.col("pred.moderate_default").alias("actual_default_30d"),
    F.col("pred.serious_default").alias("actual_default_60d"),
    
    # ========================================================================
    # PAYMENT BEHAVIOR MEASURES (Recent)
    # ========================================================================
    F.col("feat.on_time_rate_recent"),
    F.col("feat.avg_days_late_recent"),
    F.col("feat.late_rate_recent"),
    F.col("feat.max_days_late_recent"),
    F.col("feat.txn_count_recent"),
    F.col("feat.avg_order_value_recent"),
    F.col("feat.total_value_recent"),
    
    # ========================================================================
    # PAYMENT BEHAVIOR MEASURES (Medium-term)
    # ========================================================================
    F.col("feat.on_time_rate_medium"),
    F.col("feat.avg_days_late_medium"),
    F.col("feat.late_rate_medium"),
    F.col("feat.txn_count_medium"),
    F.col("feat.avg_order_value_medium"),
    
    # ========================================================================
    # PAYMENT BEHAVIOR MEASURES (Lifetime)
    # ========================================================================
    F.col("feat.on_time_rate_lifetime"),
    F.col("feat.avg_days_late_lifetime"),
    F.col("feat.max_days_late_lifetime"),
    F.col("feat.txn_count_lifetime"),
    F.col("feat.avg_order_value_lifetime"),
    F.col("feat.total_value_lifetime"),
    
    # ========================================================================
    # BEHAVIORAL TREND MEASURES
    # ========================================================================
    F.col("feat.payment_deterioration_ratio"),
    F.col("feat.late_rate_change"),
    F.col("feat.txn_velocity_ratio"),
    F.col("feat.on_time_improvement"),
    F.col("feat.payment_consistency"),
    
    # ========================================================================
    # CUSTOMER ENGAGEMENT MEASURES
    # ========================================================================
    F.col("feat.customer_tenure_days"),
    F.col("feat.days_since_last_order"),
    F.col("feat.avg_orders_per_month"),
    F.col("feat.unique_categories"),
    F.col("feat.credit_utilization"),
    
    # ========================================================================
    # FLAGS (Binary Indicators)
    # ========================================================================
    F.when(F.col("pred.predicted_tier") == F.col("pred.risk_tier"), 1).otherwise(0).alias("tier_match_flag"),
    F.when(F.col("pred.predicted_tier").isin(["Platinum", "Gold"]), 1).otherwise(0).alias("approved_tier_flag"),
    F.when(F.col("pred.predicted_tier").isin(["Platinum", "Gold", "Silver"]), 1).otherwise(0).alias("growth_tier_flag"),
    F.when(F.col("lim.final_credit_limit") > 0, 1).otherwise(0).alias("credit_granted_flag"),
    F.when(F.col("feat.payment_deterioration_ratio") > 1.5, 1).otherwise(0).alias("deteriorating_flag"),
    F.when(F.col("feat.days_since_last_order") > 60, 1).otherwise(0).alias("inactive_flag"),
    F.when(F.col("feat.on_time_rate_lifetime") >= 0.95, 1).otherwise(0).alias("excellent_payer_flag"),
)

# Save consolidated fact table
fact_credit_scoring.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_fact_credit_scoring")

print(f"✓ Created FACT_CREDIT_SCORING with {fact_credit_scoring.count():,} records")
print(f"  - Grain: One row per retailer per observation date")
print(f"  - Measures: 50+ credit scoring and behavioral metrics")
print(f"  - Dimensions: Date, Retailer, Risk Tier, Credit Product")

# ============================================================================
# CREATE SUMMARY METRICS TABLE FOR KPI CARDS
# ============================================================================

print("\n" + "=" * 80)
print("CREATING SUMMARY METRICS")
print("=" * 80)

# Calculate aggregations first
summary_agg = fact_credit_scoring.agg(
    F.countDistinct("retailer_key").alias("total_retailers"),
    F.sum("approved_tier_flag").alias("approved_retailers"),
    F.sum("final_credit_limit").alias("total_credit_exposure"),
    F.sum("expected_loss").alias("total_expected_loss"),
    F.avg("prediction_error").alias("avg_prediction_error"),
    F.sum("tier_match_flag").alias("tier_matches"),
    F.count("*").alias("total_records"),
    F.sum("actual_default_30d").alias("actual_defaults"),
    F.avg("on_time_rate_lifetime").alias("avg_on_time_rate"),
    F.avg("avg_days_late_lifetime").alias("avg_days_late"),
    F.sum(F.when(F.col("risk_tier_key") == 1, 1).otherwise(0)).alias("platinum_count"),
    F.sum(F.when(F.col("risk_tier_key") == 2, 1).otherwise(0)).alias("gold_count"),
    F.sum(F.when(F.col("risk_tier_key") == 3, 1).otherwise(0)).alias("silver_count"),
    F.sum(F.when(F.col("risk_tier_key") == 4, 1).otherwise(0)).alias("bronze_count"),
    F.sum(F.when(F.col("risk_tier_key") == 5, 1).otherwise(0)).alias("copper_count"),
)

# Add calculated columns
summary_metrics = summary_agg.select(
    F.lit(datetime.now().strftime("%Y-%m-%d")).alias("snapshot_date"),
    F.col("total_retailers"),
    F.col("approved_retailers"),
    F.col("total_credit_exposure"),
    F.col("total_expected_loss"),
    F.col("avg_prediction_error"),
    (F.col("tier_matches") / F.col("total_records")).alias("tier_accuracy"),
    F.col("actual_defaults"),
    F.col("avg_on_time_rate"),
    F.col("avg_days_late"),
    F.col("platinum_count"),
    F.col("gold_count"),
    F.col("silver_count"),
    F.col("bronze_count"),
    F.col("copper_count"),
)

summary_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_summary_metrics")

print("✓ Created SUMMARY_METRICS")

# ============================================================================
# CREATE STAR SCHEMA DOCUMENTATION
# ============================================================================

print("\n" + "=" * 80)
print("STAR SCHEMA SUMMARY")
print("=" * 80)

schema_doc = """
STAR SCHEMA STRUCTURE (Simplified - Single Fact Table):

DIMENSIONS (4):
  1. DIM_DATE - Date dimension (date_key)
  2. DIM_RETAILER - Retailer attributes (retailer_key)
  3. DIM_RISK_TIER - Risk tier definitions (tier_key)
  4. DIM_CREDIT_PRODUCT - Credit product tiers (product_key)

FACT TABLE (1):
  FACT_CREDIT_SCORING - Consolidated credit scoring fact table
     - Grain: One row per retailer per observation date
     - Links to: DIM_DATE, DIM_RETAILER, DIM_RISK_TIER, DIM_CREDIT_PRODUCT
     - Key Measure Groups:
       * Credit Scores (actual, predicted, error)
       * Credit Limits (base, final, multipliers)
       * Risk Metrics (default rates, expected loss)
       * Payment Behavior (recent, medium, lifetime)
       * Behavioral Trends (deterioration, velocity, improvement)
       * Customer Engagement (tenure, orders, utilization)
       * Flags (tier match, approval status, alerts)

BENEFITS OF SINGLE FACT TABLE:
  ✓ Simpler model structure
  ✓ Easier report development
  ✓ All metrics in one place
  ✓ No complex joins needed
  ✓ Better Power BI performance
"""

print(schema_doc)

# ============================================================================
# VALIDATE STAR SCHEMA
# ============================================================================

print("\n" + "=" * 80)
print("STAR SCHEMA VALIDATION")
print("=" * 80)

print("\nTable Row Counts:")
print(f"  DIM_DATE: {spark.table('gold_dim_date').count():,}")
print(f"  DIM_RETAILER: {spark.table('gold_dim_retailer').count():,}")
print(f"  DIM_RISK_TIER: {spark.table('gold_dim_risk_tier').count():,}")
print(f"  DIM_CREDIT_PRODUCT: {spark.table('gold_dim_credit_product').count():,}")
print(f"  FACT_CREDIT_SCORING: {spark.table('gold_fact_credit_scoring').count():,}")

# Test relationships
print("\nValidating Relationships:")

fact_table = spark.table("gold_fact_credit_scoring")

# Test 1: All fact retailers exist in dimension
fact_retailers = fact_table.select("retailer_key").distinct().count()
dim_retailers = dim_retailer.count()
print(f"  ✓ Retailer Coverage: {fact_retailers:,} fact / {dim_retailers:,} dimension")

# Test 2: All fact dates exist in dimension
fact_dates = fact_table.select("date_key").distinct().count()
dim_dates = dim_date.count()
print(f"  ✓ Date Coverage: {fact_dates:,} fact / {dim_dates:,} dimension")

# Test 3: All tier keys are valid
invalid_tiers = fact_table.join(
    dim_risk_tier, 
    fact_table.risk_tier_key == dim_risk_tier.tier_key,
    "left_anti"
).count()
print(f"  ✓ Invalid Tier Keys: {invalid_tiers} (should be 0)")

# Test 4: Check measure completeness
print("\nMeasure Completeness Check:")
null_checks = fact_table.select(
    (F.count("*") - F.count("predicted_credit_score")).alias("null_credit_scores"),
    (F.count("*") - F.count("final_credit_limit")).alias("null_credit_limits"),
    (F.count("*") - F.count("expected_loss")).alias("null_expected_loss"),
)
null_checks.show(truncate=False)

print("\n" + "=" * 80)
print("✅ STAR SCHEMA CREATION COMPLETE")
print("=" * 80)
print("\nTables created in Gold layer:")
print("  DIMENSIONS:")
print("    - gold_dim_date")
print("    - gold_dim_retailer")
print("    - gold_dim_risk_tier")
print("    - gold_dim_credit_product")
print("\n  FACT TABLE:")
print("    - gold_fact_credit_scoring (50+ measures)")
print("\n  SUMMARY:")
print("    - gold_summary_metrics")
print("\nReady for Power BI connection!")
print("=" * 80)

StatementMeta(, 51121d42-1930-4cdb-9cab-5b7b886ca6e5, 3, Finished, Available, Finished)

BUILDING STAR SCHEMA SEMANTIC MODEL FOR CREDIT SCORING

Loading source data...

CREATING DIM_DATE
✓ Created DIM_DATE with 170 records

CREATING DIM_RETAILER
✓ Created DIM_RETAILER with 10,000 records

CREATING DIM_RISK_TIER
✓ Created DIM_RISK_TIER with 5 records

CREATING DIM_CREDIT_PRODUCT (Lookup Dimension)
✓ Created DIM_CREDIT_PRODUCT with 5 records
  Note: Lookup dimension - use final_credit_limit ranges for filtering

CREATING FACT_CREDIT_SCORING (Consolidated Fact Table)
✓ Created FACT_CREDIT_SCORING with 4,668 records
  - Grain: One row per retailer per observation date
  - Measures: 50+ credit scoring and behavioral metrics
  - Dimensions: Date, Retailer, Risk Tier, Credit Product

CREATING SUMMARY METRICS
✓ Created SUMMARY_METRICS

STAR SCHEMA SUMMARY

STAR SCHEMA STRUCTURE (Simplified - Single Fact Table):

DIMENSIONS (4):
  1. DIM_DATE - Date dimension (date_key)
  2. DIM_RETAILER - Retailer attributes (retailer_key)
  3. DIM_RISK_TIER - Risk tier definitions (tier_key)
  4.

In [1]:
# ============================================================================
# REBUILD FACT TABLE WITH ALL OBSERVATION DATES
# Fix: Include ALL dates from predictions table, not just test set
# ============================================================================

from pyspark.sql import functions as F

print("=" * 80)
print("REBUILDING FACT TABLE WITH ALL OBSERVATION DATES")
print("=" * 80)

# ============================================================================
# LOAD ALL SOURCE TABLES
# ============================================================================

print("\nLoading source tables...")

predictions = spark.table("gold_credit_score_predictions")
credit_limits = spark.table("gold_credit_limits")
features = spark.table("gold_credit_scoring_features")
dim_risk_tier = spark.table("gold_dim_risk_tier")

# Show what we're working with
print(f"\nSource data:")
print(f"  Predictions: {predictions.count():,} records")
print(f"  Credit Limits: {credit_limits.count():,} records")
print(f"  Features: {features.count():,} records")

predictions_dates = predictions.select("observation_date").distinct().count()
print(f"  Observation dates in predictions: {predictions_dates}")

# ============================================================================
# BUILD COMPLETE FACT TABLE
# ============================================================================

print("\nBuilding fact table with ALL observation dates...")

# Join ALL data (not filtered by test/train split)
fact_credit_scoring = predictions.alias("pred") \
    .join(credit_limits.alias("lim"), 
          ["retailer_id", "observation_date"], "inner") \
    .join(features.select(
            "retailer_id", "observation_date",
            "on_time_rate_recent", "on_time_rate_medium", "on_time_rate_lifetime",
            "avg_days_late_recent", "avg_days_late_medium", "avg_days_late_lifetime",
            "late_rate_recent", "late_rate_medium",
            "max_days_late_recent", "max_days_late_lifetime",
            "txn_count_recent", "txn_count_medium", "txn_count_lifetime",
            "avg_order_value_recent", "avg_order_value_medium", "avg_order_value_lifetime",
            "total_value_recent", "total_value_lifetime",
            "customer_tenure_days", "days_since_last_order",
            "payment_deterioration_ratio", "late_rate_change",
            "txn_velocity_ratio", "on_time_improvement",
            "payment_consistency", "credit_utilization",
            "unique_categories", "avg_orders_per_month"
          ).alias("feat"),
          ["retailer_id", "observation_date"], "inner")

print(f"✓ After joins: {fact_credit_scoring.count():,} records")

# Create surrogate key
fact_credit_scoring = fact_credit_scoring.withColumn(
    "fact_key",
    F.monotonically_increasing_id()
)

# Map to risk tier key
fact_credit_scoring = fact_credit_scoring.join(
    dim_risk_tier.select("tier_name", "tier_key"),
    F.col("pred.predicted_tier") == F.col("tier_name"),
    "left"
).withColumnRenamed("tier_key", "risk_tier_key")

# Map to credit product key
fact_credit_scoring = fact_credit_scoring.withColumn(
    "credit_product_key",
    F.when(F.col("final_credit_limit") == 0, None)
     .when(F.col("final_credit_limit") <= 50000, 1)
     .when(F.col("final_credit_limit") <= 150000, 2)
     .when(F.col("final_credit_limit") <= 500000, 3)
     .when(F.col("final_credit_limit") <= 1000000, 4)
     .when(F.col("final_credit_limit") > 1000000, 5)
     .otherwise(None)
)

# Select final columns (same as before, but with ALL dates)
fact_credit_scoring = fact_credit_scoring.select(
    # Keys
    F.col("fact_key"),
    F.col("pred.retailer_id").alias("retailer_key"),
    F.col("pred.observation_date").alias("date_key"),
    F.col("risk_tier_key"),
    F.col("credit_product_key"),
    
    # Credit Score Measures
    F.col("pred.credit_score").alias("actual_credit_score"),
    F.col("pred.predicted_score").alias("predicted_credit_score"),
    F.abs(F.col("pred.credit_score") - F.col("pred.predicted_score")).alias("prediction_error"),
    
    # Credit Limit Measures
    F.col("lim.base_credit_limit"),
    F.col("lim.final_credit_limit"),
    F.col("lim.total_multiplier"),
    F.col("lim.history_multiplier"),
    F.col("lim.payment_bonus"),
    F.col("lim.maturity_bonus"),
    F.col("lim.shop_type_multiplier"),
    F.col("lim.location_multiplier"),
    
    # Risk Measures
    F.col("lim.expected_default_rate"),
    F.col("lim.expected_loss"),
    F.col("pred.moderate_default").alias("actual_default_30d"),
    F.col("pred.serious_default").alias("actual_default_60d"),
    
    # Payment Behavior - Recent
    F.col("feat.on_time_rate_recent"),
    F.col("feat.avg_days_late_recent"),
    F.col("feat.late_rate_recent"),
    F.col("feat.max_days_late_recent"),
    F.col("feat.txn_count_recent"),
    F.col("feat.avg_order_value_recent"),
    F.col("feat.total_value_recent"),
    
    # Payment Behavior - Medium
    F.col("feat.on_time_rate_medium"),
    F.col("feat.avg_days_late_medium"),
    F.col("feat.late_rate_medium"),
    F.col("feat.txn_count_medium"),
    F.col("feat.avg_order_value_medium"),
    
    # Payment Behavior - Lifetime
    F.col("feat.on_time_rate_lifetime"),
    F.col("feat.avg_days_late_lifetime"),
    F.col("feat.max_days_late_lifetime"),
    F.col("feat.txn_count_lifetime"),
    F.col("feat.avg_order_value_lifetime"),
    F.col("feat.total_value_lifetime"),
    
    # Behavioral Trends
    F.col("feat.payment_deterioration_ratio"),
    F.col("feat.late_rate_change"),
    F.col("feat.txn_velocity_ratio"),
    F.col("feat.on_time_improvement"),
    F.col("feat.payment_consistency"),
    
    # Customer Engagement
    F.col("feat.customer_tenure_days"),
    F.col("feat.days_since_last_order"),
    F.col("feat.avg_orders_per_month"),
    F.col("feat.unique_categories"),
    F.col("feat.credit_utilization"),
    
    # Flags
    F.when(F.col("pred.predicted_tier") == F.col("pred.risk_tier"), 1).otherwise(0).alias("tier_match_flag"),
    F.when(F.col("pred.predicted_tier").isin(["Platinum", "Gold"]), 1).otherwise(0).alias("approved_tier_flag"),
    F.when(F.col("pred.predicted_tier").isin(["Platinum", "Gold", "Silver"]), 1).otherwise(0).alias("growth_tier_flag"),
    F.when(F.col("lim.final_credit_limit") > 0, 1).otherwise(0).alias("credit_granted_flag"),
    F.when(F.col("feat.payment_deterioration_ratio") > 1.5, 1).otherwise(0).alias("deteriorating_flag"),
    F.when(F.col("feat.days_since_last_order") > 60, 1).otherwise(0).alias("inactive_flag"),
    F.when(F.col("feat.on_time_rate_lifetime") >= 0.95, 1).otherwise(0).alias("excellent_payer_flag"),
)

# ============================================================================
# VALIDATE BEFORE SAVING
# ============================================================================

print("\n" + "=" * 80)
print("VALIDATION")
print("=" * 80)

final_count = fact_credit_scoring.count()
final_dates = fact_credit_scoring.select("date_key").distinct().count()

print(f"\nFinal fact table:")
print(f"  Total records: {final_count:,}")
print(f"  Observation dates: {final_dates}")
print(f"  Unique retailers: {fact_credit_scoring.select('retailer_key').distinct().count():,}")

print("\nBreakdown by date:")
fact_credit_scoring.groupBy("date_key").agg(
    F.count("*").alias("records"),
    F.countDistinct("retailer_key").alias("unique_retailers"),
    F.sum("final_credit_limit").alias("total_exposure")
).orderBy("date_key").show(20, False)

# Check if we recovered missing dates
if final_dates >= predictions_dates:
    print(f"\n✓ SUCCESS: All {predictions_dates} observation dates included!")
else:
    print(f"\n⚠️ WARNING: Only {final_dates} of {predictions_dates} dates in fact table")
    print("   Some records may have been dropped during joins")

# ============================================================================
# SAVE
# ============================================================================

print("\n" + "=" * 80)
print("SAVING UPDATED FACT TABLE")
print("=" * 80)

fact_credit_scoring.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_fact_credit_scoring")

print("✓ Saved to gold_fact_credit_scoring")

# ============================================================================
# UPDATE SUMMARY METRICS
# ============================================================================

print("\nUpdating summary metrics...")

summary_agg = fact_credit_scoring.agg(
    F.countDistinct("retailer_key").alias("total_retailers"),
    F.sum("approved_tier_flag").alias("approved_retailers"),
    F.sum("final_credit_limit").alias("total_credit_exposure"),
    F.sum("expected_loss").alias("total_expected_loss"),
    F.avg("prediction_error").alias("avg_prediction_error"),
    F.sum("tier_match_flag").alias("tier_matches"),
    F.count("*").alias("total_records"),
    F.sum("actual_default_30d").alias("actual_defaults"),
    F.avg("on_time_rate_lifetime").alias("avg_on_time_rate"),
    F.avg("avg_days_late_lifetime").alias("avg_days_late"),
    F.sum(F.when(F.col("risk_tier_key") == 1, 1).otherwise(0)).alias("platinum_count"),
    F.sum(F.when(F.col("risk_tier_key") == 2, 1).otherwise(0)).alias("gold_count"),
    F.sum(F.when(F.col("risk_tier_key") == 3, 1).otherwise(0)).alias("silver_count"),
    F.sum(F.when(F.col("risk_tier_key") == 4, 1).otherwise(0)).alias("bronze_count"),
    F.sum(F.when(F.col("risk_tier_key") == 5, 1).otherwise(0)).alias("copper_count"),
)

from datetime import datetime

summary_metrics = summary_agg.select(
    F.lit(datetime.now().strftime("%Y-%m-%d")).alias("snapshot_date"),
    F.col("total_retailers"),
    F.col("approved_retailers"),
    F.col("total_credit_exposure"),
    F.col("total_expected_loss"),
    F.col("avg_prediction_error"),
    (F.col("tier_matches") / F.col("total_records")).alias("tier_accuracy"),
    F.col("actual_defaults"),
    F.col("avg_on_time_rate"),
    F.col("avg_days_late"),
    F.col("platinum_count"),
    F.col("gold_count"),
    F.col("silver_count"),
    F.col("bronze_count"),
    F.col("copper_count"),
)

summary_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_summary_metrics")

print("✓ Updated summary metrics")

# ============================================================================
# FINAL REPORT
# ============================================================================

print("\n" + "=" * 80)
print("✅ REBUILD COMPLETE")
print("=" * 80)

print(f"\nFact table now contains:")
print(f"  📊 {final_count:,} records")
print(f"  📅 {final_dates} observation dates")
print(f"  👥 {fact_credit_scoring.select('retailer_key').distinct().count():,} unique retailers")

print("\nNext steps:")
print("  1. Refresh your Power BI dataset")
print("  2. Verify date slicer shows all observation dates")
print("  3. Check time intelligence measures now show different values")
print("  4. Validate trends across all dates")

print("=" * 80)

StatementMeta(, 7fd9d3a6-3b30-474c-9c5c-bcfb573ca766, 3, Finished, Available, Finished)

REBUILDING FACT TABLE WITH ALL OBSERVATION DATES

Loading source tables...

Source data:
  Predictions: 6,266 records
  Credit Limits: 6,266 records
  Features: 8,150 records
  Observation dates in predictions: 8

Building fact table with ALL observation dates...
✓ After joins: 6,266 records

VALIDATION

Final fact table:
  Total records: 6,266
  Observation dates: 8
  Unique retailers: 2,109

Breakdown by date:
+----------+-------+----------------+--------------+
|date_key  |records|unique_retailers|total_exposure|
+----------+-------+----------------+--------------+
|2024-07-25|32     |32              |4773799       |
|2024-08-01|104    |104             |14295954      |
|2024-08-10|251    |251             |35426967      |
|2024-08-20|468    |468             |61459452      |
|2024-09-01|743    |743             |108961407     |
|2024-09-15|1122   |1122            |162904993     |
|2024-09-30|1540   |1540            |224046573     |
|2024-10-15|2006   |2006            |295374345     |
+